## Day 4 Homework · Cell 1 — Prepare the CPU runtime

 

Use Python 3.12 on CPU for the pinned dependencies used in this project. This cell preserves existing PyTorch or installs it if absent, then installs the specified Transformers and SentencePiece versions. Initial setup and model downloads may take longer than inference. No API key or paid inference endpoint is used. If Colab requests a restart, restart and rerun from this first cell.


In [1]:
# OPTIONAL: use a separate notebook and runtime, never the core project runtime.
# Internet and model/library downloads required; no API key or paid endpoint.
import importlib.util
import subprocess
import sys
# Preserve Colab's preinstalled matched torch/vision/audio stack.
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cpu'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'sentencepiece==0.2.0'])
import torch
print('Torch runtime:', torch.__version__)
print('If Colab requests a restart, restart this optional runtime and run all cells again.')


Torch runtime: 2.6.0
If Colab requests a restart, restart this optional runtime and run all cells again.


## Day 4 Homework · Cell 2 — Define the evaluation

This cell defines `run_evaluation_homework()`; generation starts only when the function is called in the next cell.

1. Load `google/flan-t5-small` at the pinned revision on CPU and use two CPU threads.
2. Pair each of three synthetic FAQ paraphrases with the same context: “You can check your application status using its reference number.” A supported answer should tell the user to use the reference number, without inventing fees, deadlines, or application status.
3. Convert each prompt into token IDs, generate with `do_sample=False`, `num_beams=1`, and `max_new_tokens=48`, then decode the answer.
4. Record actual input/output token counts and `inference_seconds`. Timing begins after tokenization and includes generation, decoding, and small reporting overhead; it excludes setup and model loading.
5. Check whether both “reference” and “number” appear in the answer. This keyword indicator is a weak proxy, not semantic accuracy: a wrong answer can contain both words.
6. Preserve all three cases and write `murshid_live_outputs/evaluation_result.json`.

Input text → tokenizer → token IDs → model → generated token IDs → decoded answer. Read every answer against the context rather than trusting the keyword score alone.


In [2]:
# Independent real-model evaluation homework solution. Same pinned dependencies as demo.
import json
import time
from pathlib import Path


def run_evaluation_homework():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    model_id = 'google/flan-t5-small'
    revision = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    context = 'You can check your application status using its reference number.'
    questions = ['How can I check my application status?',
                 'What do I need to check my application status?',
                 'How can I find out the status of my application?']
    rows = []
    for case_number, question in enumerate(questions, 1):
        prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
        start = time.perf_counter()
        with torch.inference_mode():
            tokens = model.generate(**inputs, max_new_tokens=48, do_sample=False, num_beams=1)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True).strip()
        assert answer, 'Empty generation; inspect runtime.'
        # A deliberately weak measurable proxy, not a semantic judge.
        keyword_hit = 'reference' in answer.lower() and 'number' in answer.lower()
        rows.append({'id': f'E{case_number:02d}', 'question': question, 'answer': answer, 'reference_keyword_hit': keyword_hit,
                     'input_tokens': int(inputs['input_ids'].shape[-1]),
                     'output_tokens_including_special': int(tokens.shape[-1]),
                     'inference_seconds': round(time.perf_counter() - start, 3)})
    result = {'mode': 'real_local_model', 'model': model_id, 'model_revision': revision,
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48},
              'metric': 'reference+number keyword proxy; NOT semantic accuracy',
              'keyword_hits': sum(x['reference_keyword_hit'] for x in rows), 'total': len(rows),
              'external_inference_api_calls': 0, 'cases': rows,
              'manual_review': 'Check every answer for correct meaning and unsupported facts. Record failures; do not edit expected criteria to force a pass.'}
    out = Path('murshid_live_outputs'); out.mkdir(exist_ok=True)
    (out / 'evaluation_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result, indent=2))
    return result




## Day 4 Homework · Cell 3 — Run and review the three answers

Calls the evaluation function, loads the model, generates E01–E03, and prints and saves the report. The first model download may take time; later runs can reuse downloaded weights.

Check `mode: "real_local_model"`, `total: 3`, `external_inference_api_calls: 0`, the generation settings, and each case’s question, answer, tokens, and measured time. The instructor observed three keyword hits, but keep your actual results even if answers or timings differ.

For each answer, ask: Does it convey that a reference number is needed? Does it add unsupported facts? Record your interpretation and any failures. A feasible improvement is a larger, independently reviewed set with negative examples; three related English questions cannot establish general accuracy.

Download `evaluation_result.json` from the `murshid_live_outputs` folder and save this executed notebook separately. The recorded review below describes this saved run; update it if you rerun and obtain different answers.


In [3]:
result = run_evaluation_homework()


{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "generation": {
    "do_sample": false,
    "num_beams": 1,
    "max_new_tokens": 48
  },
  "metric": "reference+number keyword proxy; NOT semantic accuracy",
  "keyword_hits": 3,
  "total": 3,
  "external_inference_api_calls": 0,
  "cases": [
    {
      "id": "E01",
      "question": "How can I check my application status?",
      "answer": "Use its reference number",
      "reference_keyword_hit": true,
      "input_tokens": 34,
      "output_tokens_including_special": 6,
      "inference_seconds": 2.444
    },
    {
      "id": "E02",
      "question": "What do I need to check my application status?",
      "answer": "Reference number",
      "reference_keyword_hit": true,
      "input_tokens": 36,
      "output_tokens_including_special": 4,
      "inference_seconds": 0.02
    },
    {
      "id": "E03",
      "question": "How can I find out the statu

## Review of the recorded run

These are the actual outputs generated on CPU for this saved notebook. The interpretation below was prepared with AI assistance; review it before submission and update it if a later run differs.

| Case | Actual answer | Input / output tokens | Inference seconds | Meaning review |
|---|---|---|---|---|
| E01 | Use its reference number | 34 / 6 | 2.444 | The answer tells the user to use the reference number, matching the supplied context. It adds no fee, date, or application status. |
| E02 | Reference number | 36 / 4 | 0.02 | The short noun phrase directly answers what is needed: the reference number. It omits a full sentence but adds no unsupported requirement. |
| E03 | Use its reference number | 37 / 6 | 0.026 | The answer correctly points to the reference number as the way to check status, with no unsupported details. |

All three answers contain both keywords and are supported by the supplied context on inspection. No generation failure or unsupported claim was observed in these three cases. The keyword count of 3/3 is not a semantic-accuracy score.

**Limitation:** this small, closely related English set misses many failure modes. **Improvement:** add independently reviewed negative and ambiguous examples, including answers that mention the reference number but make an unsupported claim.

The local run used Python 3.12.7, PyTorch 2.6.0, Transformers 4.49.0, SentencePiece 0.2.0, CPU, and two threads. Previously downloaded weights were reused offline. External inference API calls and billed API cost were zero. Timings are observations, not performance targets.
